# 18-1. LangGraph 대조

선택 심화. `18`을 마친 뒤 본다.

같은 과제를 LangGraph 로 옮겨 자작 하네스, Agents SDK 와 나란히 둔다. 그래프로 옮기면 무엇이 드러나고 무엇이 새로 생기는지 본다.
</br>Modular RAG : 각 노드를 AGENT로 취급 (파이썬 함수)

### 공통 설정

`18`과 같은 블록이다. 세 구현의 사용량을 한 곳에 모으려고 `note`를 하나 더 둔다. SDK 와 LangGraph 는 응답 객체를 `keep`에 넘기지 않기 때문이다.

In [1]:
import asyncio
import inspect
import json
import os
import time
from pathlib import Path
from types import SimpleNamespace

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI(max_retries=6)

PRICES = {MODEL: (2.50, 15.00)}  # 1M 토큰당 (입력, 출력) USD
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp


def note(model, input_tokens, output_tokens):
    USAGE.append((model, SimpleNamespace(input_tokens=input_tokens, output_tokens=output_tokens)))

코퍼스와 도구 함수다. 과제는 둘이다. 하나는 `18`과 같고, 하나는 검색을 두 번 해서 겹치는 문서를 세야 한다. 정답은 코드로 센다.

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")


def grep_docs(pattern):
    return [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]


def both(a, b):
    return len({d["title"] for d in grep_docs(a)} & {d["title"] for d in grep_docs(b)})


TASKS = [("'한국독립당' 이 나오는 문서가 몇 편인지 세고 숫자로 답한다", len(grep_docs("한국독립당"))),
         ("'한국독립당' 과 '조선민족혁명당' 을 모두 언급한 문서가 몇 편인지 세고 숫자로 답한다",
          both("한국독립당", "조선민족혁명당"))]
TASK = TASKS[0][0]
print([n for _, n in TASKS])

[16, 6]


### 자작 하네스

`18`의 루프를 그대로 옮긴다. 줄 수를 세려고 구현 하나를 `build_*` 함수 하나에 담는다. 돌려주는 `run`은 답, 모델 호출 수, 토큰을 낸다.
`effort`는 뒤에서 세 구현에 같은 추론 설정을 줄 때 쓴다. 지금은 비워 둔다.

In [3]:
def build_own(effort=None):
    TOOLS = [{"type": "function", "name": "grep_docs", "strict": True,
              "description": "낱말이 든 문서의 제목과 등장 횟수를 돌려준다.",
              "parameters": {"type": "object",
                             "properties": {"pattern": {"type": "string", "description": "찾을 낱말"}},
                             "required": ["pattern"], "additionalProperties": False}}]
    CONFIG = {"instructions": "도구로 확인한 것만 답한다.", "max_turns": 6,
              "max_tokens": 60000, "tool_error": "error: {e}"}

    def call_tool(name, args):
        try:
            return json.dumps(grep_docs(**args), ensure_ascii=False)
        except Exception as e:
            return CONFIG["tool_error"].format(e=e)

    extra = {"reasoning": {"effort": effort}} if effort else {}

    def run(task):
        messages, used = [{"role": "user", "content": task}], 0
        for turn in range(1, CONFIG["max_turns"] + 1):
            r = keep(client.responses.create(model=MODEL, instructions=CONFIG["instructions"],
                                             input=messages, tools=TOOLS, store=False, **extra))
            used += r.usage.total_tokens
            calls = [i for i in r.output if i.type == "function_call"]
            if not calls:
                return r.output_text, turn, used
            if used > CONFIG["max_tokens"]:
                return None, turn, used
            messages += [i.model_dump(exclude_none=True) for i in r.output]
            for c in calls:
                messages.append({"type": "function_call_output", "call_id": c.call_id,
                                 "output": call_tool(c.name, json.loads(c.arguments))})
        return None, CONFIG["max_turns"], used

    return run


run_own = build_own()
print(run_own(TASK))

('16', 2, 503)


### Agents SDK

`18`의 `counter` 에이전트다. 사용량은 `result.context_wrapper.usage`에 모여 있다. `18`에서 감춰졌다고 한 사용량은 여기서 꺼낼 수 있다.

In [ ]:
from agents import Agent, ModelSettings, Runner, function_tool
from openai.types.shared import Reasoning


def build_sdk(effort=None, settings=None):
    @function_tool
    def grep_docs_tool(pattern: str) -> list[dict]:
        """낱말이 든 문서의 제목과 등장 횟수를 돌려준다."""
        return grep_docs(pattern)

    if effort:
        settings = ModelSettings(reasoning=Reasoning(effort=effort))    # Reasoning effort를 주기위해 ModelSettings 활용
    counter = Agent(name="counter", model=MODEL, **({"model_settings": settings} if settings else {}),
                    instructions="도구로 확인한 것만 답한다.", tools=[grep_docs_tool])

    async def run(task):
        result = await Runner.run(counter, task, max_turns=6)
        u = result.context_wrapper.usage
        note(MODEL, u.input_tokens, u.output_tokens)
        return result.final_output, u.requests, u.total_tokens

    return run


run_sdk = build_sdk()
print(await run_sdk(TASK))

('16', 2, 512)


### LangGraph 로 옮긴다

문법은 1주차 `19`에서 본 세 가지다. 상태는 `TypedDict`, 노드는 상태를 받아 바꿀 부분만 돌려주는 함수, 엣지는 다음 노드를 정한다.
루프는 모델 노드와 도구 노드 사이의 엣지 하나가 된다. 도구 호출이 없으면 조건 엣지가 `END`로 보낸다.

`messages`의 `add_messages`는 덮어쓰지 않고 이어 붙이라는 표시다. `tokens`는 토큰 예산을 상태에 두려고 넣었다.

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# Agentic하게 쓰려면 이렇게 정의해야함!
class State(TypedDict):
    messages: Annotated[list, add_messages] # 출력되는 메시지가 추가되어(add_messags) list로 저장 (agentic loop 작동방식과 동일 / 추가만 되고 삭제되지 않음ㄴ)
    tokens: int


@tool   # tool 어노테이션 : 사용할 수 있는 도구가 됨
def grep_docs_lc(pattern: str) -> list[dict]:
    """낱말이 든 문서의 제목과 등장 횟수를 돌려준다.""" # 도구에 대한 설명
    return grep_docs(pattern)


def make_llm(effort=None):
    extra = {"reasoning": {"effort": effort}} if effort else {}
    return ChatOpenAI(model=MODEL, use_responses_api=True, max_retries=6, **extra).bind_tools([grep_docs_lc])   # 도구를 가진 llm (bind_tools)

노드 둘과 라우터 하나다. 모델 노드는 모델을 바꿔 끼울 수 있게 만드는 함수로 둔다.
도구 노드는 `langgraph.prebuilt.ToolNode`로 대신할 수 있지만, 오류를 문자열로 돌려주는 자리를 보이려고 직접 쓴다.

In [ ]:
def make_model_node(llm):
    def model_node(state):
        msg = llm.invoke([SystemMessage("도구로 확인한 것만 답한다.")] + state["messages"])
        u = msg.usage_metadata
        note(MODEL, u["input_tokens"], u["output_tokens"])
        return {"messages": [msg], "tokens": state["tokens"] + u["total_tokens"]}
    return model_node


model_node = make_model_node(make_llm())


def tool_node(state):
    out = []
    for c in state["messages"][-1].tool_calls:  # -1 : 마지막 메시지에 tool_calls가 포함
        try:
            body = json.dumps(grep_docs(**c["args"]), ensure_ascii=False)   # tool을 호출함
        except Exception as e:
            body = f"error: {e}"
        out.append(ToolMessage(body, tool_call_id=c["id"]))
    return {"messages": out}


def route(state, max_tokens=60000):
    if not state["messages"][-1].tool_calls or state["tokens"] > max_tokens:
        return END  # 종료처리
    return "tools"  # tools로 라우트

그래프를 조립한다. `recursion_limit`은 노드 실행 횟수의 한도다. 모델과 도구를 한 번씩 돌면 2가 늘어나므로 반복 6턴은 12다.

In [7]:
def build_graph(tools=tool_node, checkpointer=None, model=model_node):
    g = StateGraph(State)
    g.add_node("model", model)
    g.add_node("tools", tools)
    g.add_edge(START, "model")
    g.add_conditional_edges("model", route, ["tools", END])
    g.add_edge("tools", "model")
    return g.compile(checkpointer=checkpointer)


def make_run_graph(graph):
    def run(task):
        out = graph.invoke({"messages": [HumanMessage(task)], "tokens": 0}, {"recursion_limit": 12})
        calls = sum(1 for m in out["messages"] if m.type == "ai")
        return out["messages"][-1].text, calls, out["tokens"]
    return run


app = build_graph()
run_graph = make_run_graph(app)
print(run_graph(TASK))

('16', 2, 499)


컴파일한 그래프는 그림으로 확인한다. 자작 루프의 for 문이 `model`과 `tools` 사이의 고리로 보인다.
</br>https://www.processon.io/mermaid?gad_source=1&gad_campaignid=21726972931&gbraid=0AAAAA-PYQFZsRRUjvkjNsWOGWt_Oj75_y&gclid=Cj0KCQjwlsfWBhD4ARIsACL7p1EKSlnQGINJa6Ac2WEwe7G7wdkIDv7EAx0MvK07wxbmRVl_blBdyswaAv2aEALw_wcB 여기에 붙여넣으면 머메이드 그림으로 확인 가능

In [8]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools --> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### 세 구현을 같은 과제로

과제마다 다섯 번씩 돌린다. 정답 여부, 모델 호출 수, 토큰을 모은다.

In [9]:
async def call(impl, task):
    out = impl(task)
    return await out if inspect.isawaitable(out) else out


RUNS = {"자작": run_own, "SDK": run_sdk, "LangGraph": run_graph}
rows = []
for name, impl in RUNS.items():
    for task, truth in TASKS:
        for _ in range(5):
            answer, calls, tokens = await call(impl, task)
            ok = str(truth) in str(answer).replace(",", "").split("편")[0]
            rows.append({"구현": name, "과제": TASKS.index((task, truth)) + 1,
                         "정답": ok, "호출": calls, "토큰": tokens})

for name in RUNS:
    for t in (1, 2):
        rs = [r for r in rows if r["구현"] == name and r["과제"] == t]
        print(f"{name:10s} 과제{t}  정답 {sum(r['정답'] for r in rs)}/5"
              f"   호출 {sum(r['호출'] for r in rs) / 5:.1f}   토큰 {sum(r['토큰'] for r in rs) / 5:7,.0f}")

자작         과제1  정답 5/5   호출 2.0   토큰     503
자작         과제2  정답 5/5   호출 2.0   토큰     741
SDK        과제1  정답 5/5   호출 2.0   토큰     512
SDK        과제2  정답 0/5   호출 2.0   토큰     753
LangGraph  과제1  정답 5/5   호출 2.0   토큰     499
LangGraph  과제2  정답 3/5   호출 2.0   토큰     743


`build_*` 함수의 줄 수다. 빈 줄은 빼고 센다. LangGraph 쪽은 도구, 모델, 노드, 라우터, 조립 함수를 세고, 상태 정의는 필드 수에 한 줄을 더한다.
노트북 안에서 정의한 클래스는 `inspect`가 소스를 찾지 못하기 때문이다.

In [10]:
def lines(*objs):
    return sum(len([l for l in inspect.getsource(o).splitlines() if l.strip()]) for o in objs)


print("자작      ", lines(build_own))
print("SDK       ", lines(build_sdk))
print("LangGraph ", lines(grep_docs_lc.func, make_llm, make_model_node, tool_node, route, build_graph, make_run_graph)
      + len(State.__annotations__) + 1)

자작       31
SDK        15
LangGraph  44


코드의 양은 SDK 가 가장 짧다. LangGraph 는 루프를 노드와 엣지로 풀어 쓰고 모델과 그래프를 바꿔 끼우는 함수까지 두어서 자작 하네스보다 길다.

호출 수는 셋이 같다. 과제 1 은 셋 다 맞는다. 과제 2 는 구현에 따라 정답 수가 갈리고, 다시 돌리면 어느 구현이 틀리는지도 바뀐다. 도구 결과는 같고 모델이 센 숫자만 다르다.
구현의 실력 차이처럼 보이지만 원인은 다른 데 있다. 세 구현은 같은 모델에 같은 요청을 보내고 있지 않다.

### 프레임워크가 보내는 요청

세 구현 모두 안에서는 OpenAI 클라이언트로 요청을 보낸다. 클라이언트가 HTTP 요청을 보내는 `request`를 잠시 감싸서 실제로 나가는 본문을 모은다.
`request`는 내부 함수라 클라이언트 버전이 바뀌면 이름이 달라질 수 있다. 과제 1 을 한 번씩 돌리고 첫 요청만 본다.
SDK 는 `ModelSettings()`를 빈 채로 직접 준 것도 하나 더 넣는다.

In [11]:
from openai._base_client import AsyncAPIClient, SyncAPIClient

SENT = {}
_req, _areq = SyncAPIClient.request, AsyncAPIClient.request


def spy(self, cast_to, options, **kw):
    if options.url == "/responses":
        SENT.setdefault(CURRENT, options.json_data)
    return _req(self, cast_to, options, **kw)


async def aspy(self, cast_to, options, **kw):
    if options.url == "/responses":
        SENT.setdefault(CURRENT, options.json_data)
    return await _areq(self, cast_to, options, **kw)


SyncAPIClient.request, AsyncAPIClient.request = spy, aspy
try:
    for CURRENT, impl in {**RUNS, "SDK 설정 지정": build_sdk(settings=ModelSettings())}.items():
        await call(impl, TASK)
finally:
    SyncAPIClient.request, AsyncAPIClient.request = _req, _areq

추론 설정, 출력 길이 설정, 도구 이름, 인자 스키마, strict 여부를 나란히 찍는다. 넣지 않은 값은 `-`다.

In [12]:
for name, body in SENT.items():
    t = body["tools"][0]
    effort = (body.get("reasoning") or {}).get("effort", "-")
    verbosity = (body.get("text") or {}).get("verbosity", "-")
    print(f"{name:14s} reasoning {effort:5s} verbosity {verbosity:4s}"
          f" strict {str(t.get('strict', '-')):5s} 도구 {t['name']}")
    print(f"{'':14s} 인자 {json.dumps(t['parameters']['properties'], ensure_ascii=False)}")

자작             reasoning -     verbosity -    strict True  도구 grep_docs
               인자 {"pattern": {"type": "string", "description": "찾을 낱말"}}
SDK            reasoning none  verbosity low  strict True  도구 grep_docs_tool
               인자 {"pattern": {"title": "Pattern", "type": "string"}}
LangGraph      reasoning -     verbosity -    strict -     도구 grep_docs_lc
               인자 {"pattern": {"type": "string"}}
SDK 설정 지정      reasoning -     verbosity -    strict True  도구 grep_docs_tool
               인자 {"pattern": {"title": "Pattern", "type": "string"}}


자작 하네스는 적은 것만 보낸다. SDK 는 gpt-5 계열 모델에 추론 effort `none`과 출력 길이 `low`를 기본으로 넣고, 독스트링에서 만든 인자 스키마에는 설명이 없다.
`ModelSettings()`를 빈 채로라도 직접 주면 이 기본값이 빠진다. 설정을 주었는지 아닌지로 요청이 달라진다.
LangChain 은 strict 를 빼고 인자 설명 없이 스키마를 만든다.

하나씩 보면 무해해 보인다. 그런데 과제 2 처럼 추론 없이 겨우 맞히는 과제에서는 이런 차이가 정답률을 바꾼다.
이 노트북을 준비하며 자작 하네스에 SDK 의 도구 정의와 출력 길이 `low`를 하나씩 얹었을 때는 정답률이 그대로였고, 둘을 함께 얹었을 때 크게 떨어졌다. 도구 이름 하나를 바꿔도 결과가 갈렸다.
어느 조합이 왜 그런지는 모델 안의 일이라 알 수 없다. 알 수 있는 것은 프레임워크의 기본값이 요청에 들어가 있다는 사실이고, 그것은 이렇게 꺼내 봐야 보인다.

### 같은 설정을 주면

세 구현에 추론 effort `low`를 똑같이 주고 과제 2 를 다섯 번씩 다시 돌린다.

In [13]:
LOW = {"자작": build_own("low"), "SDK": build_sdk("low"),
       "LangGraph": make_run_graph(build_graph(model=make_model_node(make_llm("low"))))}
task, truth = TASKS[1]
for name, impl in LOW.items():
    oks = []
    for _ in range(5):
        answer, calls, tokens = await call(impl, task)
        oks.append(str(truth) in str(answer).replace(",", "").split("편")[0])
    print(f"{name:10s} 과제2  정답 {sum(oks)}/5")

자작         과제2  정답 5/5
SDK        과제2  정답 5/5
LangGraph  과제2  정답 5/5


추론을 조금 주면 구현 사이의 차이가 거의 사라진다. **구현을 비교하기 전에 같은 요청을 보내는지부터 맞춘다.** 프레임워크를 바꾸면 성능이 바뀌었다는 보고를 들으면 먼저 요청을 꺼내 본다.

### 정지 조건

반복 한도는 SDK 처럼 예외로 온다. `recursion_limit`을 넘으면 `GraphRecursionError`가 난다.

In [14]:
from langgraph.errors import GraphRecursionError

try:
    app.invoke({"messages": [HumanMessage(TASKS[1][0])], "tokens": 0}, {"recursion_limit": 2})
except GraphRecursionError as e:
    print("반복 한도 |", type(e).__name__)

반복 한도 | GraphRecursionError


토큰 예산은 다르다. 상태에 누적 토큰이 있으므로 라우터가 읽고 `END`로 보낼 수 있다. SDK 에서는 실행 바깥에서 처리해야 했던 것이 그래프 안의 엣지 하나가 된다.
예산을 아주 작게 준 라우터로 같은 그래프를 다시 만든다. 첫 모델 호출 뒤 도구를 부르지 못하고 끝난다.

In [15]:
def tight(state):
    return route(state, max_tokens=50)


g = StateGraph(State)
g.add_node("model", model_node)
g.add_node("tools", tool_node)
g.add_edge(START, "model")
g.add_conditional_edges("model", tight, ["tools", END])
g.add_edge("tools", "model")
out = g.compile().invoke({"messages": [HumanMessage(TASK)], "tokens": 0})
last = out["messages"][-1]
print("토큰", out["tokens"], "| 남은 도구 호출", len(last.tool_calls), "| 답", repr(last.text))

토큰 116 | 남은 도구 호출 1 | 답 ''


예산으로 멈추면 마지막 메시지가 답이 아니라 도구 호출이다. 무엇이 멈췄는지는 상태를 보고 판단해야 한다. 자작 하네스가 정지 사유를 따로 돌려준 것과 같은 일을 여기서도 직접 해야 한다.

### 체크포인터

컴파일할 때 체크포인터를 주면 **노드가 끝날 때마다 상태가 저장**된다. 저장 단위는 `thread_id`다.
메모리에 두는 `InMemorySaver`로 본다. 노드 하나가 끝날 때마다 스냅숏이 하나씩 쌓인다.

In [16]:
from langgraph.checkpoint.memory import InMemorySaver

saved = build_graph(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "count-1"}, "recursion_limit": 12}
saved.invoke({"messages": [HumanMessage(TASK)], "tokens": 0}, cfg)

for snap in reversed(list(saved.get_state_history(cfg))):
    print(f"  다음 {str(snap.next):12s} 메시지 {len(snap.values.get('messages', []))}개  토큰 {snap.values.get('tokens', 0)}")

  다음 ('__start__',) 메시지 0개  토큰 0
  다음 ('model',)   메시지 1개  토큰 0
  다음 ('tools',)   메시지 2개  토큰 116
  다음 ('model',)   메시지 3개  토큰 116
  다음 ()           메시지 4개  토큰 499


저장이 쓸모 있는 것은 중간에 실패했을 때다. 처음 한 번은 예외를 던지는 도구 노드를 넣는다.
실패한 뒤 같은 `thread_id`로 `None`을 넣어 다시 부르면 마지막 체크포인트에서 이어 간다. 이미 끝난 모델 호출은 다시 하지 않는다.

In [17]:
state = {"fail": True}


def flaky_tools(st):
    if state["fail"]:
        state["fail"] = False
        raise ConnectionError("도구 서버가 끊겼다")
    return tool_node(st)


resumable = build_graph(tools=flaky_tools, checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "count-2"}, "recursion_limit": 12}
before = len(USAGE)
try:
    resumable.invoke({"messages": [HumanMessage(TASK)], "tokens": 0}, cfg)
except ConnectionError as e:
    print("실패 |", e, "| 다음 노드", resumable.get_state(cfg).next, "| 모델 호출", len(USAGE) - before)

out = resumable.invoke(None, cfg)
print("재개 |", out["messages"][-1].text, "| 모델 호출 누적", len(USAGE) - before)

실패 | 도구 서버가 끊겼다 | 다음 노드 ('tools',) | 모델 호출 1
재개 | 16 | 모델 호출 누적 2


`22`에서는 같은 일을 입력 배열과 턴 번호를 파일 하나에 저장해서 직접 한다. 여기서는 그 저장과 재개가 컴파일 옵션 하나다.
대신 무엇이 언제 저장되는지가 노드 경계로 정해진다. 노드 안에서 도구를 여러 번 부르다 실패하면 그 노드는 처음부터 다시 돈다.

### interrupt 로 도구 단위 승인

`18`의 선택 기준에서 도구 단위 승인은 직접 구현할 것이었다. LangGraph 에서는 노드 안에서 `interrupt`를 부르면 그 자리에서 멈추고 값을 밖으로 내보낸다.
승인 결과는 `Command(resume=...)`로 넣는다. 체크포인터가 있어야 멈춘 자리를 기억한다.

노드에 들어온 횟수를 같이 센다.

In [ ]:
from langgraph.types import Command, interrupt

entered = []


def gated_tools(st):
    entered.append(1)
    calls = st["messages"][-1].tool_calls
    decision = interrupt({"승인 요청": [(c["name"], c["args"]) for c in calls]})
    if decision != "승인":
        return {"messages": [ToolMessage("error: 사람이 거절했다. 확인하지 못했다고 답한다.", tool_call_id=c["id"]) #tool call의 결과가 toolMessage로 감
                             for c in calls]}
    return tool_node(st)


gated = build_graph(tools=gated_tools, checkpointer=InMemorySaver())

실행하면 도구 노드에서 멈춘다. 반환값의 `__interrupt__`에 승인 요청이 들어 있다.

In [19]:
cfg = {"configurable": {"thread_id": "gate-1"}, "recursion_limit": 12}
r = gated.invoke({"messages": [HumanMessage(TASK)], "tokens": 0}, cfg)
print(r["__interrupt__"][0].value)
print("다음 노드", gated.get_state(cfg).next, "| 노드 진입", len(entered))

{'승인 요청': [('grep_docs_lc', {'pattern': '한국독립당'})]}
다음 노드 ('tools',) | 노드 진입 1


승인하면 도구가 돌고 답이 나온다.

In [20]:
out = gated.invoke(Command(resume="승인"), cfg)
print(out["messages"][-1].text, "| 노드 진입", len(entered))

16 | 노드 진입 2


노드 진입이 두 번이다. 재개하면 노드를 `interrupt` 자리부터가 아니라 처음부터 다시 실행한다. 그래서 `interrupt` 앞에 부작용이 있는 코드를 두면 두 번 실행된다.
거절하는 쪽도 본다. 도구 결과 대신 거절 문자열을 받은 모델이 어떻게 답하는지 확인한다.
거절 문자열도 Part 1 의 오류 메시지처럼 복구 지시문이다. "도구 없이 답한다"라고 쓰면 모델이 숫자를 지어낼 수 있어서, 확인하지 못했다고 답하라고 적는다.

In [21]:
cfg = {"configurable": {"thread_id": "gate-2"}, "recursion_limit": 12}
gated.invoke({"messages": [HumanMessage(TASK)], "tokens": 0}, cfg)
out = gated.invoke(Command(resume="거절"), cfg)
print(out["messages"][-1].text[:120])

확인하지 못했습니다.


### 구성 요소의 대응

| 자작 하네스 | Agents SDK | LangGraph |
|---|---|---|
| Runtime (`run`의 for 문) | `Runner.run` | 모델 노드와 도구 노드 사이의 엣지, `invoke` |
| ToolRegistry (`TOOLS`, `call_tool`) | `@function_tool` | `@tool`과 `bind_tools`, 도구 노드 |
| Session (`messages` 조립) | `Session`, `to_input_list()` | 상태의 `messages`와 리듀서 `add_messages` |
| Trace (`trace` 리스트) | `result.new_items`, 대시보드 | `get_state_history`의 스냅숏 |
| Policy (설정의 게이트) | 실행 전후 가드레일 | 노드 안의 `interrupt` |
| 정지 조건 셋 | `max_turns` 예외 | `recursion_limit` 예외, 상태를 읽는 라우터 |
| 중단과 재개 | 없음 | 체크포인터와 `thread_id` |
| 요청 옵션 | 적은 것만 보낸다 | effort, verbosity 기본값을 넣는다 | 적은 것만 보낸다. strict 와 인자 설명은 빠진다 |

SDK 는 루프를 감추고 LangGraph 는 루프를 그래프로 드러낸다. 그래서 LangGraph 에서는 자작 하네스에서 하던 일을 대부분 같은 자리에서 할 수 있고, 저장과 재개, 승인처럼 직접 만들기 번거로운 것을 거저 얻는다.
대신 상태가 노드 경계에서만 저장되고 재개하면 노드가 처음부터 다시 돈다는 실행 모델을 알고 써야 한다.
</br>
보통 랭그래프 구현하여 가능여부를 확인하구 => 추후 자작하네스로 (범위가 넓으므로)

### 선택 기준

| 조건 | 선택 |
|---|---|
| 루프가 하나이고 반복 한도면 충분하다 | SDK |
| 실행 흐름에 분기(conditional edge가 여러개)와 고리(loop도는 것)가 여럿 있다 + subgraph 지원 | LangGraph |
| 중간 저장과 재개, 도구 단위 승인이 필요하다 | LangGraph, 또는 직접 구현 |
| 컨텍스트를 턴마다 직접 고쳐야 한다 | 직접 구현, 또는 LangGraph 노드에서 상태를 고친다 |
| 평가 코드에 맞는 스팬과 정지 사유가 필요하다 | 직접 구현 (랭그래프의 runnable을 수정해야해서 번거롭) |

어느 쪽을 고르든 프레임워크가 넣는 기본값은 버전마다 바뀐다. 버전을 올리면 요청을 한 번 꺼내 본다.

### 비용 확인

세 구현의 사용량을 모두 합산한다. SDK 와 LangGraph 는 `note`로 넣은 것이다.

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")